# Visualizations Ablation Analysis:

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import numpy as np
from tqdm import tqdm
import os

In [5]:
from visualize_method_performance import create_performance_plots, create_phenotype_specific_plots

## Data loading and cleaning:

In [2]:
metadata_df = pd.read_csv('collected_metadata_pathological_phens.csv')

In [3]:
metadata_df.dropna(subset=["num_snps_found"], inplace=True)

In [4]:
metadata_df.rename(columns={'description':'phenotype'}, inplace=True)

In [6]:
create_performance_plots(metadata_df, output_folder_name='vis_patho_runtime')

In [16]:
print("metadata_df['num_original_list'].median()")
print(metadata_df['num_original_list'].median())
print("metadata_df['num_original_list'].max()")
print(metadata_df['num_original_list'].max())
print("metadata_df['num_overlapping_loci'].median()")
print(metadata_df['num_overlapping_loci'].median())
print("metadata_df['num_overlapping_loci'].max()")
print(metadata_df['num_overlapping_loci'].max())
print("metadata_df['num_overlapping_snps'].median()")
print(metadata_df['num_overlapping_snps'].median())
print("metadata_df['num_overlapping_snps'].max()")
print(metadata_df['num_overlapping_snps'].max())

metadata_df['num_original_list'].median()
0.0
metadata_df['num_original_list'].max()
481.0
metadata_df['num_overlapping_loci'].median()
0.0
metadata_df['num_overlapping_loci'].max()
478.0
metadata_df['num_overlapping_snps'].median()
0.0
metadata_df['num_overlapping_snps'].max()
48.0


In [7]:
create_phenotype_specific_plots(metadata_df, output_folder_name='vis_patho_runtime')

## Visualization (ablations grouped by alpha):

In [15]:
alpha_dfs = metadata_df.groupby('alpha')

for alpha, sub_df in alpha_dfs:
    create_performance_plots(sub_df, alpha=alpha)


In [16]:
alpha_dfs = metadata_df.groupby('alpha')

for alpha, sub_df in alpha_dfs:
    create_phenotype_specific_plots(sub_df, alpha=alpha)


## Compare averages:

In [17]:
alpha_dfs = metadata_df.groupby('alpha')
sumstats_list =[]
max_snp_df = pd.DataFrame(np.zeros([len(metadata_df), 6]),
                          columns=[f'max_snps_{alpha}' for alpha in [0.01, 0.025, 0.05, 0.1, 0.25, 0.5]])

for alpha, sub_df in alpha_dfs:
    sumstats_list.append({
        'Alpha': alpha,
        'Average SNP Recovery Rate': sub_df['num_overlapping_snps'].mean() / sub_df['num_original_list'].mean(),
        'Average Loci Recovery Rate': sub_df['num_overlapping_loci'].mean() / sub_df['num_original_list'].mean(),
        'Average Additional SNPs Found': (sub_df['num_snps_found'] - sub_df['num_overlapping_snps']).mean(),
        'Average Additional Coding SNPs': (sub_df['num_coding_snps_found'] - sub_df['num_original_coding_snps']).mean(),
        'Number of Phenotypes': len(sub_df),
        'Median SNP Recovery Rate': sub_df['num_overlapping_snps'].median() / sub_df['num_original_list'].median(),
        'Median Loci Recovery Rate': sub_df['num_overlapping_loci'].median() / sub_df['num_original_list'].median(),
        'Average Relative Increase': ((sub_df['num_snps_found'] - sub_df['num_original_list']) / sub_df['num_original_list']).mean(),
        'Median Relative Increase': ((sub_df['num_snps_found'] - sub_df['num_original_list']) / sub_df['num_original_list']).median(),
        # 'Max_SNPs_discovered': (sub_df['num_snps_found'] - sub_df['num_overlapping_snps']).max()
    })
    # max_snp_df[f'max_snps_{alpha}'] = sub_df['num_snps_found'] - sub_df['num_overlapping_snps']

sumstats_df = pd.DataFrame(sumstats_list)

In [ ]:

relative_max = []

for idx, row in max_snp_df.iterrows():
    max_of_row = row.max()
    for alpha in [0.01, 0.025, 0.05, 0.1, 0.25, 0.5]:
        row[f'perc_max_snps_{alpha}'] = row[f'max_snps_{alpha}'] / max_of_row

for alpha in [0.01, 0.025, 0.05, 0.1, 0.25, 0.5]:
    relative_max.append(max_snp_df[f'max_snps_{alpha}'].mean())

sumstats_df['Additional SNPs Found relative to Max'] = relative_max
# sumstats_df['Additional SNPs Found relative to Max'] = sumstats_df['Average Additional SNPs Found'] / sumstats_df['Max_SNPs_discovered'].max()

In [18]:
sumstats_df

,Alpha,Average SNP Recovery Rate,Average Loci Recovery Rate,Average Additional SNPs Found,Average Additional Coding SNPs,Number of Phenotypes,Median SNP Recovery Rate,Median Loci Recovery Rate,Average Relative Increase,Median Relative Increase
0,0.5,0.326278,0.967333,225.155556,38.229630,135,0.230769,0.923077,inf,-0.127936
1,1.0,0.190890,0.945999,249.360294,72.426471,136,0.126582,0.822785,inf,-0.214463
2,1.5,0.146764,0.929143,239.411765,93.257353,136,0.113924,0.784810,inf,-0.273653
3,2.0,0.124678,0.916227,230.573529,108.485294,136,0.101266,0.746835,inf,-0.310861
4,2.5,0.112257,0.904675,223.595588,120.007353,136,0.088608,0.721519,inf,-0.330759
5,3.0,0.103117,0.894531,218.750000,128.691176,136,0.075949,0.670886,inf,-0.333333


In [19]:
sumstats_df.to_csv("visualization_output/sumstats.csv")